In [0]:
catalog = "ecommerce_project"
schema  = "sales"
path    = f"/Volumes/{catalog}/{schema}/raw_files"

display(dbutils.fs.ls(path))

In [0]:
import re
from pyspark.sql import functions as F

catalog = "ecommerce_project"
schema  = "sales"
path    = f"/Volumes/{catalog}/{schema}/raw_files"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

def to_snake_case(df):
    """Rename columns like 'Order ID' -> 'order_id', 'Sub-Category' -> 'sub_category'."""
    for c in df.columns:
        new = re.sub(r"[^0-9a-zA-Z]+", "_", c.strip()).strip("_").lower()
        df = df.withColumnRenamed(c, new)
    return df

In [0]:
%run ./00_transformations

In [0]:
orders_raw = to_snake_case(
    spark.read.option("multiline", "true").json(f"{path}/Orders.json")
)
orders_raw.printSchema()
save_table(orders_raw, "raw_orders", expected_min_rows=9994)


In [0]:
products_raw = to_snake_case(
    spark.read.option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("inferSchema", "true")
    .csv(f"{path}/Products.csv")
)
products_raw.printSchema()
save_table(products_raw, "raw_products", expected_min_rows=1851, overwrite_schema=True)


In [0]:
p = spark.table("raw_products")
p.printSchema()                                   # price_per_product should be double
print(p.count())                                  # 1851
print(p.filter(F.col("price_per_product").isNull()).count())          # 0
print(p.select("product_id").distinct().count())  # 1818
print(p.filter(F.col("product_name").contains('"')).count())          # product names with real quotes should look intact, e.g. 6" Cubicle Wall Clock

In [0]:
# Pure PySpark: Spark's built-in Excel reader (no pandas)
customers_xlsx = (
    spark.read.format("excel")
    .option("headerRows", 1)
    .load(f"{path}/Customer.xlsx")
)
# keep all columns as text, same as the CSV version
customers_raw = to_snake_case(
    customers_xlsx.select([F.col(f"`{c}`").cast("string").alias(c) for c in customers_xlsx.columns])
)
customers_raw.printSchema()
save_table(customers_raw, "raw_customers", expected_min_rows=793)


In [0]:
for t in ["raw_orders", "raw_products", "raw_customers"]:
    print(t, spark.table(t).count())
    display(spark.table(t).limit(3))